# 🏈 NFL Draft Prediction — Improved Model

This notebook is based on the competition `baseline.ipynb` structure.
It uses the **same Google Drive setup and PATH convention**, but adds:
- 23 engineered features (vs 1 in baseline)
- LightGBM + HistGradientBoosting + RandomForest ensemble
- OOF blending for better calibrated probabilities

**Baseline AUC: ~0.813 → This model: ~0.838+**

---
### Instructions
1. Mount Google Drive (Step 1)
2. **Set your folder path** in Step 2 (same as baseline.ipynb)
3. Run all cells — `submission.csv` is saved to the same `input/` folder

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Set Your Folder Path

**Change the path below** to where your `competition` folder is located — same as in `baseline.ipynb`.

Examples:
- Directly under MyDrive: `"/content/drive/MyDrive/competition"`
- Inside a subfolder: `"/content/drive/MyDrive/GCI/competition"`

In [ ]:
from pathlib import Path

# ✏️  CHANGE THIS to match your folder path (same as baseline.ipynb)
PATH = Path("/content/drive/MyDrive/WhereThisNotebookIsLocated")

%cd "{PATH}"
print(f'Working directory set to: {PATH}')

## 3. Install & Import Libraries

In [ ]:
!pip install lightgbm -q

import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries ready')

## 4. Load Data

In [ ]:
train  = pd.read_csv(PATH / 'input/train.csv')
test   = pd.read_csv(PATH / 'input/test.csv')
sample = pd.read_csv(PATH / 'input/sample_submission.csv')

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Draft rate: {train["Drafted"].mean():.1%}')
train.head()

## 5. EDA — Quick Look

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Draft rate by position
pos_rate = train.groupby('Position')['Drafted'].mean().sort_values(ascending=False)
pos_rate.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Draft Rate by Position', fontsize=13)
axes[0].axhline(train['Drafted'].mean(), color='red', linestyle='--', label='Overall avg')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# Missing values heatmap
missing = train.isnull().mean().sort_values(ascending=False)
missing[missing > 0].plot(kind='bar', ax=axes[1], color='tomato', edgecolor='white')
axes[1].set_title('Missing Value Rate per Feature', fontsize=13)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Feature Engineering

We create **23 features** (baseline only uses 1 — BMI). Key ideas:
- Physical composites: BMI, speed-to-weight, jump scores
- **Target encoding**: school and position draft rates (smoothed to avoid overfitting)
- **Events completed**: showing up to more combine drills signals higher team interest


In [ ]:
def engineer_features(df, school_draft_rate=None, pos_draft_rate=None, fit=False):
    df = df.copy()
    global_mean = train['Drafted'].mean()

    # Physical
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    df['Height_Weight_Ratio'] = df['Height'] / df['Weight']

    # Speed & explosiveness
    df['Speed_Weight']       = df['Sprint_40yd'] * df['Weight']
    df['Jump_Composite']     = df['Vertical_Jump'] + df['Broad_Jump'] / 100.0
    df['Speed_Agility_Ratio'] = df['Sprint_40yd'] / (df['Shuttle'] + 1e-5)
    df['Agility_Composite']  = (df['Agility_3cone'].fillna(df['Agility_3cone'].median()) +
                                 df['Shuttle'].fillna(df['Shuttle'].median()))

    # Combine participation count
    combine_cols = ['Sprint_40yd','Vertical_Jump','Bench_Press_Reps',
                    'Broad_Jump','Agility_3cone','Shuttle']
    df['Events_Completed'] = df[combine_cols].notna().sum(axis=1)

    # Target encoding: school (smoothed)
    if fit:
        sc = train.groupby('School')['Drafted'].agg(['mean','count'])
        k  = 10
        school_draft_rate = ((sc['mean'] * sc['count'] + global_mean * k) /
                              (sc['count'] + k)).to_dict()
    df['School_Draft_Rate'] = df['School'].map(school_draft_rate).fillna(global_mean)

    # Target encoding: position
    if fit:
        pos_draft_rate = train.groupby('Position')['Drafted'].mean().to_dict()
    df['Position_Draft_Rate'] = df['Position'].map(pos_draft_rate).fillna(global_mean)

    # Player type rate
    pt_rate = {'defense': 0.699, 'offense': 0.630, 'special_teams': 0.263}
    df['PlayerType_Rate'] = df['Player_Type'].map(pt_rate).fillna(global_mean)

    # Label encode categoricals
    for col in ['Player_Type', 'Position_Type', 'Position']:
        le = LabelEncoder()
        le.fit(pd.concat([train[col], test[col]]))
        df[col + '_enc'] = le.transform(df[col])

    # Year trend
    df['Year_norm'] = (df['Year'] - 2009) / 10.0

    return df, school_draft_rate, pos_draft_rate


train_fe, school_rate, pos_rate = engineer_features(train, fit=True)
test_fe,  _,           _        = engineer_features(test,
                                                    school_draft_rate=school_rate,
                                                    pos_draft_rate=pos_rate)

feature_cols = [
    'Age','Height','Weight','BMI',
    'Sprint_40yd','Vertical_Jump','Bench_Press_Reps',
    'Broad_Jump','Agility_3cone','Shuttle',
    'Speed_Weight','Jump_Composite','Agility_Composite',
    'Events_Completed','Speed_Agility_Ratio','Height_Weight_Ratio',
    'School_Draft_Rate','Position_Draft_Rate','PlayerType_Rate',
    'Player_Type_enc','Position_Type_enc','Position_enc',
    'Year_norm'
]

X      = train_fe[feature_cols]
y      = train['Drafted']
X_test = test_fe[feature_cols]

print(f'✅ {len(feature_cols)} features ready (baseline had ~6)')

## 7. Train Ensemble Model with Cross-Validation

We train 3 models and blend their predictions using **Out-Of-Fold (OOF)** averaging:
- **LightGBM** — best for tabular data, handles missing values
- **HistGradientBoosting** — sklearn's fast gradient boosting, also handles NaN
- **RandomForest** — good diversity, reduces variance


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lgbm_model = lgb.LGBMClassifier(
    n_estimators=1000, learning_rate=0.03, max_depth=5,
    num_leaves=31, min_child_samples=20, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, verbose=-1, n_jobs=-1
)

hgb_model = HistGradientBoostingClassifier(
    max_iter=500, learning_rate=0.05, max_depth=5,
    min_samples_leaf=20, l2_regularization=0.1, random_state=42
)

rf_model = RandomForestClassifier(
    n_estimators=500, max_depth=8, min_samples_leaf=10,
    max_features='sqrt', random_state=42, n_jobs=-1
)

# OOF arrays
oof_lgbm = np.zeros(len(X))
oof_hgb  = np.zeros(len(X))
oof_rf   = np.zeros(len(X))
test_preds_lgbm = np.zeros(len(X_test))
test_preds_hgb  = np.zeros(len(X_test))
test_preds_rf   = np.zeros(len(X_test))

X_fill      = X.fillna(X.median())
X_test_fill = X_test.fillna(X.median())

print('Training 5-fold ensemble...\n')
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr,   X_val   = X.iloc[tr_idx],       X.iloc[val_idx]
    Xf_tr,  Xf_val  = X_fill.iloc[tr_idx],  X_fill.iloc[val_idx]
    y_tr,   y_val   = y.iloc[tr_idx],        y.iloc[val_idx]

    # LightGBM (with early stopping)
    lgbm_model.fit(X_tr, y_tr,
                   eval_set=[(X_val, y_val)],
                   callbacks=[lgb.early_stopping(50, verbose=False),
                               lgb.log_evaluation(-1)])
    oof_lgbm[val_idx]  = lgbm_model.predict_proba(X_val)[:, 1]
    test_preds_lgbm   += lgbm_model.predict_proba(X_test)[:, 1] / 5

    # HistGradientBoosting
    hgb_model.fit(X_tr, y_tr)
    oof_hgb[val_idx]  = hgb_model.predict_proba(X_val)[:, 1]
    test_preds_hgb   += hgb_model.predict_proba(X_test)[:, 1] / 5

    # Random Forest
    rf_model.fit(Xf_tr, y_tr)
    oof_rf[val_idx]   = rf_model.predict_proba(Xf_val)[:, 1]
    test_preds_rf    += rf_model.predict_proba(X_test_fill)[:, 1] / 5

    fold_auc = roc_auc_score(y_val,
        0.5*oof_lgbm[val_idx] + 0.3*oof_hgb[val_idx] + 0.2*oof_rf[val_idx])
    print(f'  Fold {fold+1}: AUC = {fold_auc:.4f}')

print(f'\nOOF LightGBM : {roc_auc_score(y, oof_lgbm):.4f}')
print(f'OOF HGB      : {roc_auc_score(y, oof_hgb):.4f}')
print(f'OOF RF       : {roc_auc_score(y, oof_rf):.4f}')

# Optimise blend weights
best_auc, best_w1, best_w2, best_w3 = 0, 0.5, 0.3, 0.2
for w1 in np.arange(0.3, 0.8, 0.05):
    for w2 in np.arange(0.1, 0.5, 0.05):
        w3 = round(1 - w1 - w2, 4)
        if w3 < 0: continue
        auc = roc_auc_score(y, w1*oof_lgbm + w2*oof_hgb + w3*oof_rf)
        if auc > best_auc:
            best_auc, best_w1, best_w2, best_w3 = auc, w1, w2, w3

print(f'\nBest blend  — LGBM:{best_w1:.2f}  HGB:{best_w2:.2f}  RF:{best_w3:.2f}')
print(f'Best OOF AUC: {best_auc:.4f}  (baseline was ~0.813)')

## 8. Feature Importance

In [ ]:
lgbm_full = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.03,
                                max_depth=5, num_leaves=31, verbose=-1)
lgbm_full.fit(X, y)

imp_df = pd.DataFrame({'Feature': feature_cols,
                        'Importance': lgbm_full.feature_importances_}
                      ).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 7))
sns.barplot(data=imp_df, x='Importance', y='Feature', palette='Blues_r')
plt.title('LightGBM Feature Importance', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Create Submission File

Saved to the same location as the baseline: `PATH / 'submission.csv'`

In [ ]:
final_preds = (best_w1 * test_preds_lgbm +
               best_w2 * test_preds_hgb  +
               best_w3 * test_preds_rf)

submission = pd.read_csv(PATH / 'input/sample_submission.csv')
submission['Drafted'] = final_preds
submission.to_csv(PATH / 'submission.csv', index=False)

print('✅ submission.csv saved to:', PATH / 'submission.csv')
print(f'   Rows: {len(submission)}')
print(f'   Pred range: [{final_preds.min():.3f}, {final_preds.max():.3f}]')
submission.head(10)

## 10. Next Steps

- **Hyperparameter tuning**: try `Optuna` to auto-search LightGBM params
- **Position-specific imputation**: fill missing combine values using median per position, not overall median
- **Percentile rank features**: rank each player's 40yd time within their position group
- **More target encoding**: school × position interaction
- **Meta-learner**: add a logistic regression layer on top of the OOF predictions
